### Import modules and data

In [ ]:
import pandas as pd
from pathlib import Path
import sys

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS
from backend.data.cleaning import clean_string_series

In [ ]:
reservoirs_path = PATHS['raw_data_notebooks'] / 'reservoirs.csv'
reservoirs_pd = pd.read_csv(reservoirs_path)
reservoirs_pd.head()

In [ ]:
reservoirs_pd.info()

### Renaming columns

In [ ]:
columns_dict = {'ID': 'id', 'SCOPE_NAME': 'scope', 'RESERVOIR_NAME': 'name', 'TOTAL_WATER':'capacity', 'ELECTRIC_FLAG': 'electric_flag'}
reservoirs_pd.rename(columns=columns_dict, inplace=True)

### Looking for Missing Values before cleaning texts and converting dtypes

In [ ]:
reservoirs_pd.isna().sum()

### Converting capacity to Integer

In [ ]:
# Let's ensure that every float is actually an int
print(f"Number of rows: {len(reservoirs_pd)}")
print(f"Number of int at capacity: {reservoirs_pd['capacity'].dropna().apply(float.is_integer).sum()}")

In [ ]:
reservoirs_pd.loc[reservoirs_pd['capacity'].notna(), 'capacity'] = reservoirs_pd.loc[reservoirs_pd['capacity'].notna(), 'capacity'].astype(int)

### Text Cleaning

In [ ]:
reservoirs_pd.head()

In [ ]:
reservoirs_pd['name'] = clean_string_series(reservoirs_pd['name'])
reservoirs_pd['scope'] = clean_string_series(reservoirs_pd['scope'])

### Save current cleaning and developing reservoir.ipynb at EDA folder

In [ ]:
cleaned_reservoir_path = PATHS['pre_EDA'] / 'reservoir_for_EDA.csv'
cleaned_reservoir_path.parent.mkdir(parents=True, exist_ok=True)
reservoirs_pd.to_csv(cleaned_reservoir_path, index=False)


### Handling Missing Values

In [ ]:
water_pd = pd.read_csv(PATHS['cleaned_data_notebooks'] / 'water_cleaned.csv')

# As it's the capacity of the reservoir, we will fill it with the maximum registered value
id_to_current_water = water_pd.groupby('id')['storage'].max() # Creates a series with 'ID' as index
mask = reservoirs_pd['capacity'].isna()
reservoirs_pd.loc[mask, 'capacity'] = reservoirs_pd.loc[mask, 'id'].map(id_to_current_water) # Replacing missing values efficiently

In [ ]:
reservoirs_pd.head()

In [ ]:
reservoirs_pd.info()

### Saving cleaned data

In [ ]:
cleaned_reservoirs_path = PATHS['cleaned_data_notebooks'] / 'reservoirs_cleaned.csv'
cleaned_reservoirs_path.parent.mkdir(parents=True, exist_ok=True)
reservoirs_pd.to_csv(cleaned_reservoirs_path, index=False)